<a href="https://colab.research.google.com/github/SteFabolous/stem-splitter-with-google-colab/blob/main/stem_splitter_with_google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# @title ⚙️ Step 1: Connect Google Drive, Install Dependencies & Setup Environment
import os
import sys
from google.colab import drive

print("⚙️ Initializing environment (Drive connection, dependencies & workspace folders)...")
drive.mount('/content/drive')

# Install dependencies and official NVIDIA CUDA 12 packages required by ONNX Runtime
!pip install -q "onnxruntime-gpu" "audio-separator[gpu]" yt-dlp spotdl ml_collections nvidia-cublas-cu12 nvidia-cudnn-cu12 > /dev/null 2>&1

# Expose NVIDIA pip library paths to LD_LIBRARY_PATH as specified in ONNX Runtime documentation
py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
nvidia_dir = f"/usr/local/lib/python{py_ver}/dist-packages/nvidia"
if os.path.exists(nvidia_dir):
    lib_paths = [os.path.join(nvidia_dir, pkg, "lib") for pkg in os.listdir(nvidia_dir) if os.path.isdir(os.path.join(nvidia_dir, pkg, "lib"))]
    os.environ['LD_LIBRARY_PATH'] = ":".join(lib_paths) + ":" + os.environ.get('LD_LIBRARY_PATH', '')

# Create workspace folders on Google Drive
input_folder = "/content/drive/MyDrive/Input_Audio"
output_folder = "/content/drive/MyDrive/Stems_Output"

os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

print("✅ Setup complete! Upload your audio files to 'Input_Audio' on Google Drive and run Step 2.")

⚙️ Initializing environment (Drive connection, dependencies & workspace folders)...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup complete! Upload your audio files to 'Input_Audio' on Google Drive and run Step 2.


In [5]:
# @title 🎛️ Step 2: Configure & Run Stem Splitter
import os
import sys

# Attempt to fix CUDA library loading issues by ensuring a compatible onnxruntime-gpu is installed
# and CUDA library paths are correctly set.

print("⚙️ Reconfiguring ONNX Runtime for CUDA compatibility...")

# Uninstall existing onnxruntime and onnxruntime-gpu to ensure a clean state.
# The -y flag answers yes to confirmation prompts.
!pip uninstall -y onnxruntime onnxruntime-gpu > /dev/null 2>&1

# Install a known good onnxruntime-gpu version (1.17.1) for CUDA 12.
# Using --force-reinstall to ensure it replaces any existing incompatible version.
# Installing from the specific ONNX Runtime CUDA 12 index.
!pip install -q onnxruntime-gpu==1.17.1 --index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ --force-reinstall > /dev/null 2>&1

# Re-expose NVIDIA pip library paths to LD_LIBRARY_PATH for the current session.
# This is critical for onnxruntime to find the CUDA shared libraries.
py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
nvidia_dir = f"/usr/local/lib/python{py_ver}/dist-packages/nvidia"
lib_paths = []
if os.path.exists(nvidia_dir):
    lib_paths = [os.path.join(nvidia_dir, pkg, "lib") for pkg in os.listdir(nvidia_dir) if os.path.isdir(os.path.join(nvidia_dir, pkg, "lib"))]

# Add standard CUDA library path if it exists
if os.path.exists('/usr/local/cuda/lib64'):
    lib_paths.append('/usr/local/cuda/lib64')

# Only set LD_LIBRARY_PATH if there are paths to add
if lib_paths:
    os.environ['LD_LIBRARY_PATH'] = ":".join(lib_paths) + ":" + os.environ.get('LD_LIBRARY_PATH', '')

print("✅ ONNX Runtime reconfigured. Attempting to proceed...")

import glob
import shutil
import urllib.request
from audio_separator.separator import Separator

# @markdown ### 📁 Audio Source (Choose one)
# @markdown Input folder path on Google Drive (containing audio files):
audio_input_folder = "/content/drive/MyDrive/Input_Audio"  # @param {type:"string"}

# @markdown Paste a YouTube URL:
youtube_url = ""  # @param {type:"string"}

# @markdown Or paste a Spotify Track/Album/Playlist URL:
spotify_url = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🤖 Select AI Model
model_choice = "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)"  # @param ["MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)", "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)", "MelBand-RoFormer BigBeta 6 (Balanced Vocals)", "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)", "MelBand-RoFormer BigBeta 4 (Classic)", "MelBand-RoFormer BigBeta 3", "BS-RoFormer Leap (unwa SOTA)", "Kim-unwa FT2 (Vocal Hybrid)", "BS-RoFormer ViperX (Vocals)", "HTDemucs v4 FT (4 Stems)", "HTDemucs v4 6-Stems (Guitar/Piano)"]

# @markdown ---
# @markdown ### ⚙️ Advanced Model & Output Parameters
# @markdown **Overlap**: Higher values improve transition quality at segment boundaries (1 to 40).
overlap = 40  # @param {type:"slider", min:1, max:40, step:1}

# @markdown **Output Format**: Audio format for saved stems.
output_format = "wav"  # @param ["wav", "flac", "mp3"]

# @markdown **Chunk Size (Segment Size)**: Processing window size. Lower values save GPU VRAM.
chunk_size_choice = "529200"  # @param ["112455", "352800", "485100", "529200", "661500"]
chunk_size = int(chunk_size_choice)

# @markdown **Use TTA (Test-Time Augmentation)**: Enables extra inversion passes for cleaner stems (doubles processing time).
use_tta = True  # @param {type:"boolean"}

# @markdown **Extract Instrumental**: Keep enabled to output both Vocals & Instrumental stems. If unchecked, outputs Vocals only.
extract_instrumental = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 💾 Main Output Folder on Google Drive
output_folder = "/content/drive/MyDrive/Stems_Output"  # @param {type:"string"}

os.makedirs(output_folder, exist_ok=True)

# Mappatura dei checkpoint ufficiali riconosciuti dal registro di audio-separator
models_map = {
    # --- Series BigBeta (by pcunwa) ---
    "MelBand-RoFormer BigBeta 7 (Latest SOTA Vocals)": "melband_roformer_big_beta7.ckpt",
    "MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)": "melband_roformer_big_beta6x.ckpt",
    "MelBand-RoFormer BigBeta 6 (Balanced Vocals)": "melband_roformer_big_beta6.ckpt",
    "MelBand-RoFormer BigBeta 5e (Max Vocal Fullness)": "melband_roformer_big_beta5e.ckpt",
    "MelBand-RoFormer BigBeta 4 (Classic)": "melband_roformer_big_beta4.ckpt",
    "MelBand-RoFormer BigBeta 3": "melband_roformer_big_beta3.ckpt",

    # --- Modelli recenti di unwa & Kimberley Jensen ---
    "BS-RoFormer Leap (unwa SOTA)": "bs_roformer_leap.ckpt",
    "Kim-unwa FT2 (Vocal Hybrid)": "kimmel_unwa_ft2.ckpt",
    "BS-RoFormer ViperX (Vocals)": "model_bs_roformer_ep_368_sdr_12.9779.ckpt",

    # --- Multi-Stem ---
    "HTDemucs v4 FT (4 Stems)": "htdemucs_ft",
    "HTDemucs v4 6-Stems (Guitar/Piano)": "htdemucs_6s"
}

selected_model = models_map[model_choice]

# Handling Input (Priority: Spotify > YouTube > Drive Folder)
files_to_process = []

if spotify_url.strip():
    print("🟢 Downloading track(s) from Spotify via SpotDL...")
    temp_spot_dir = "/content/spotdl_temp"
    if os.path.exists(temp_spot_dir):
        shutil.rmtree(temp_spot_dir)
    os.makedirs(temp_spot_dir, exist_ok=True)
    !spotdl download "{spotify_url}" --output "{temp_spot_dir}" --format wav
    files_to_process = glob.glob(f"{temp_spot_dir}/*.wav")
    if not files_to_process:
        raise FileNotFoundError("❌ Unable to download track(s) from Spotify.")

elif youtube_url.strip():
    print("📥 Downloading audio from YouTube...")
    temp_yt_file = "/content/yt_temp.wav"
    if os.path.exists(temp_yt_file):
        os.remove(temp_yt_file)
    !yt-dlp -x --audio-format wav -o "{temp_yt_file}" "{youtube_url}"
    if os.path.exists(temp_yt_file):
        files_to_process = [temp_yt_file]
    else:
        raise FileNotFoundError("❌ Unable to download audio from YouTube.")

elif audio_input_folder.strip():
    if not os.path.exists(audio_input_folder):
        raise FileNotFoundError(f"❌ Input folder not found at: {audio_input_folder}")

    audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg', '.opus', '.wma', '.aiff')
    files_to_process = [
        os.path.join(audio_input_folder, f)
        for f in os.listdir(audio_input_folder)
        if f.lower().endswith(audio_extensions)
    ]
    files_to_process.sort()

    if not files_to_process:
        raise FileNotFoundError(f"❌ No valid audio files found in: {audio_input_folder}")

print(f"\n📂 Found {len(files_to_process)} track(s) to process.")
print(f"🚀 Initializing model [{model_choice}]...")
print(f"⚙️ Config: Overlap={overlap} | Format={output_format.upper()} | Chunk={chunk_size} | TTA={use_tta} | Full Stems={extract_instrumental}\n")

# Initialize separator
separator = Separator(
    output_dir=output_folder,
    output_format=output_format.upper(),
    output_single_stem=None if extract_instrumental else "Vocals"
)

# Apply parameters
separator.roformer_overlap = overlap
separator.mdx_overlap = overlap
separator.mdxc_overlap = overlap

separator.roformer_segment_size = chunk_size
separator.mdx_segment_size = chunk_size

if use_tta:
    separator.mdx_enable_tta = True
    separator.vr_enable_tta = True

# Fallback automatico per i nomi modello se non presenti nell'indice predefinito
try:
    separator.load_model(selected_model)
except Exception:
    # Se il nome esteso non viene trovato, tenta il fallback sul nome del file base
    short_model_name = selected_model.replace("melband_roformer_", "")
    separator.load_model(short_model_name)

# Process all files
for idx, file_path in enumerate(files_to_process, 1):
    track_name = os.path.splitext(os.path.basename(file_path))[0]

    track_output_dir = os.path.join(output_folder, track_name)
    os.makedirs(track_output_dir, exist_ok=True)

    separator.output_dir = track_output_dir

    print(f"[{idx}/{len(files_to_process)}] 🎵 Processing: {track_name}...")
    output_files = separator.separate(file_path)
    print(f"   ✅ Saved stems to: `{track_output_dir}`\n")

print("🔥 ALL SEPARATIONS COMPLETED!")
print(f"📁 Check your main output directory on Google Drive: `{output_folder}`")

⚙️ Reconfiguring ONNX Runtime for CUDA compatibility...


INFO:audio_separator.separator.separator:Separator version 0.47.0 instantiating with output_dir: /content/drive/MyDrive/Stems_Output, output_format: WAV
INFO:audio_separator.separator.separator:Using model directory from model_file_dir parameter: /tmp/audio-separator-models/
INFO:audio_separator.separator.separator:Operating System: Linux #1 SMP Thu Apr 30 18:17:14 UTC 2026
INFO:audio_separator.separator.separator:System: Linux Node: 37d7a9d272a7 Release: 6.6.122+ Machine: x86_64 Proc: x86_64
INFO:audio_separator.separator.separator:Python Version: 3.13.15
INFO:audio_separator.separator.separator:PyTorch Version: 2.11.0+cpu


✅ ONNX Runtime reconfigured. Attempting to proceed...

📂 Found 1 track(s) to process.
🚀 Initializing model [MelBand-RoFormer BigBeta 6X (Ultra Clean Vocals)]...
⚙️ Config: Overlap=40 | Format=WAV | Chunk=529200 | TTA=True | Full Stems=True



INFO:audio_separator.separator.separator:FFmpeg installed: ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
INFO:audio_separator.separator.separator:No hardware acceleration could be configured, running in CPU mode
INFO:audio_separator.separator.separator:Loading model melband_roformer_big_beta6x.ckpt...


Failed to load libcublasLt.so.13: libcublasLt.so.13: cannot open shared object file: No such file or directory
Failed to load libcublas.so.13: libcublas.so.13: cannot open shared object file: No such file or directory
Failed to load libnvrtc.so.13: libnvrtc.so.13: cannot open shared object file: No such file or directory
Failed to load libcurand.so.10: libcurand.so.10: cannot open shared object file: No such file or directory
Failed to load libcufft.so.12: libcufft.so.12: cannot open shared object file: No such file or directory
Failed to load libcudart.so.13: libcudart.so.13: cannot open shared object file: No such file or directory
Please follow https://onnxruntime.ai/docs/install/#cuda-and-cudnn to install CUDA.


INFO:audio_separator.separator.separator:MDXC Separator initialisation complete
INFO:audio_separator.separator.separator:Roformer loading stats: {'new_implementation_success': 1, 'total_failures': 0}
INFO:audio_separator.separator.separator:Load model duration: 00:00:08
INFO:audio_separator.separator.separator:Processing file: /content/drive/MyDrive/Input_Audio/Cammy Barnes - Something More.flac
INFO:audio_separator.separator.separator:Starting separation process for audio_file_path: /content/drive/MyDrive/Input_Audio/Cammy Barnes - Something More.flac
INFO:audio_separator.separator.separator:Input audio subtype: PCM_24
INFO:audio_separator.separator.separator:Detected input bit depth: 24-bit


[1/1] 🎵 Processing: Cammy Barnes - Something More...


  0%|          | 0/30 [00:02<?, ?it/s]
INFO:audio_separator.separator.separator:Clearing input audio file paths, sources and stems...


KeyboardInterrupt: 

In [15]:
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu --index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

Found existing installation: onnxruntime-gpu 1.20.0
Uninstalling onnxruntime-gpu-1.20.0:
  Successfully uninstalled onnxruntime-gpu-1.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.4/475.4 MB 839.3 kB/s eta 0:00:00


# 📖 User Guide & Parameter Breakdown

---

### ⚙️ What Do the First Two Cells Do?

#### **Step 1: Setup & Environment Prep**
- **Mounts Google Drive**: Connects your Drive to `/content/drive` so the script can access your audio files and save stems directly to the cloud.
- **Installs Tooling**: Auto-installs `audio-separator` (the SOTA GPU-supported AI backend), `yt-dlp` (for YouTube ripping), and `spotdl` (for Spotify downloading).
- **Auto-creates Folders**: Generates `Input_Audio` (where you drop tracks) and `Stems_Output` (where processed stems land) directly on your Drive.

#### **Step 2: Configuration & Stem Separator Execution**
- **Audio Ingestion**: Prioritizes Spotify URL > YouTube URL > Batch processing all files inside your `Input_Audio` Drive folder.
- **Model Execution**: Loads the AI model weights into GPU memory (T4) **once** and sequentially splits tracks one after another (saving tons of render time).
- **Clean File Organization**: Automatically creates a dedicated subfolder for each track inside `Stems_Output`, keeping your Drive clean and organized.

---

### 🎛️ Detailed Parameter Guide (Step 2)

#### **1. 📁 Audio Sources**
You have three input options with a strict hierarchy:
- **Spotify URL**: Highest priority. Paste a track, album, or playlist link. Uses `spotdl` to match and download high-quality audio.
- **YouTube URL**: Second priority. Paste any YouTube link to download and process audio on the fly in WAV format.
- **Input Folder Path**: If Spotify and YouTube inputs are blank, the script scans this Drive folder and batch-processes **all** valid audio files (`.mp3`, `.wav`, `.flac`, `.m4a`, etc.) in a single run.

---

#### **2. 🤖 AI Model Selection**

- **MelBand-RoFormer BigBeta Series (by pcunwa)**:
  - **BigBeta 7**: Absolute SOTA for vocals. Ultra-clean bleed removal without flattening high-frequency air or vocal transients.
  - **BigBeta 6X**: Surgical isolation. Zero-bleed focus, perfect for isolated acapellas in mashups/bootlegs where the vocal plays solo.
  - **BigBeta 6**: The ideal sweet spot between vocal fullness and clean instrumental rejection.
  - **BigBeta 5e**: Focused on vocal warmth (`Enhanced Fullness`). Keeps the low-end warmth of the lead vocal, but might leave minor instrumental bleed if heavy synths are present.
  - **BigBeta 4 / 3**: Classic fallback models if newer versions produce weird artifacts on specific tracks.

- **BS-RoFormer Series (Leap & ViperX)**:
  - Alternative SOTA architectures. **Leap** boasts insane SDR (signal-to-distortion ratio) scores and can outperform RoFormer on complex electronic tracks with heavy synth layers.

- **HTDemucs v4 (4-Stems / 6-Stems)**:
  - Use this when you need full multitrack separation (Drums, Bass, Guitar, Piano, Other) instead of just Vocal/Instrumental split.

---

#### **3. ⚙️ Advanced Parameters**

- **Overlap (1 - 40)**:
  - Controls how many times the AI overlaps analysis windows at segment boundaries to prevent clicks or seam artifacts.
  - *Pro Tip*: Stick to **`4` to `8`**. Going over `10` exponentially increases render times with almost zero noticeable gain.

- **Output Format (WAV / FLAC / MP3)**:
  - **WAV**: Uncompressed 24/32-bit audio. Ideal for dropping straight into your DAW (FL Studio, REAPER, etc.).
  - **FLAC**: Lossless compression (same audio quality as WAV, ~50% smaller file size).
  - **MP3**: Lossy compression, only use if you're running tight on Drive storage.

- **Chunk Size (Segment Size)**:
  - Processing window size (`112455`, `352800`, `485100`, `529200`, `661500`) loaded into GPU VRAM.
  - *Pro Tip*: Standard **`352800`** or **`485100`** works great. If Colab crashes with a `CUDA Out of Memory` error (especially with heavy models or ultra-long tracks), lower it to **`112455`**.

- **Use TTA (Test-Time Augmentation)**:
  - Runs a second pass with phase inversion to catch hidden frequencies and bleed.
  - *Pros*: Slightly cleaner acapellas.
  - *Cons*: Exactly doubles the processing time per track.

- **Extract Instrumental**:
  - **Enabled (True)**: Saves both `Vocals` and `Instrumental` stems.
  - **Disabled (False)**: Outputs only the isolated vocal stem, saving processing time and storage space.

---

*✨ Project built with the help of AI.*